# Titanic — Modeling

**Purpose:** Build, evaluate, and tune machine learning models to predict survival.  
This notebook loads the engineered features from `03_feature_engineering.ipynb` — no raw data manipulation here.

**What we do:**
1. Load `train_engineered.csv` + `test_engineered.csv`
2. Preprocess with a leakage-safe `ColumnTransformer`
3. Benchmark 6 models (including LightGBM) with Stratified K-Fold
4. Compare models on Accuracy, ROC-AUC, and F1
5. Tune XGBoost with RandomizedSearchCV
6. Interpret the best model with **SHAP**
7. Generate Kaggle submission files

## 0) Setup

In [ ]:
import os
import numpy as np
import pandas as pd

# Modeling & pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                              confusion_matrix, ConfusionMatrixDisplay,
                              RocCurveDisplay)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Interpretation
import shap

# Viz
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import loguniform, randint

# ── Constants ────────────────────────────────────────────────────────────────
DATA_DIR     = "../data"
OUTPUT_DIR   = "../outputs"
RANDOM_STATE = 42
CV_FOLDS     = 5
TEST_SIZE    = 0.15

np.random.seed(RANDOM_STATE)
sns.set(style="whitegrid", context="notebook")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"pandas {pd.__version__}  |  numpy {np.__version__}")

## 1) Load engineered data

We load the output of `03_feature_engineering.ipynb` — original columns plus all engineered features already computed.  
Separating feature engineering from modeling keeps this notebook focused and easier to iterate on.

In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, "train_engineered.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test_engineered.csv"))

print("train:", train.shape, "  test:", test.shape)

TARGET = 'Survived'
ID_COL = 'PassengerId'

X          = train.drop(columns=[TARGET])
y          = train[TARGET]
X_test_raw = test.copy()

train.head(3)

## 2) Preprocessing pipeline

All preprocessing lives inside a `ColumnTransformer` so train/validation/test always get identical treatment — this is the standard way to avoid data leakage.

- **Numeric:** median imputation (handles missing `Age` / `FarePerPerson` / `WomanOrChild`)
- **Categorical:** most-frequent imputation + one-hot encoding (ignores unseen categories at test time)

In [ ]:
NUMERIC_FEATURES = [
    'Age', 'SibSp', 'Parch', 'Fare', 'Pclass',
    'FamilySize', 'Fare_log1p', 'FarePerPerson', 'TicketGroupSize', 'WomanOrChild'
]

CATEGORICAL_FEATURES = [
    'Sex', 'Embarked', 'Title', 'IsAlone', 'Deck', 'AgeGroup'
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer([
    ("num", numeric_transformer,     NUMERIC_FEATURES),
    ("cat", categorical_transformer, CATEGORICAL_FEATURES),
], remainder='drop')

print(f"Numeric features  ({len(NUMERIC_FEATURES)}): {NUMERIC_FEATURES}")
print(f"Categorical features ({len(CATEGORICAL_FEATURES)}): {CATEGORICAL_FEATURES}")

### 2.1) Inspect the transformed design matrix

In [ ]:
preprocess.fit(X)
X_transformed = preprocess.transform(X)

ohe           = preprocess.named_transformers_['cat'].named_steps['onehot']
cat_names     = ohe.get_feature_names_out(CATEGORICAL_FEATURES).tolist()
feature_names = NUMERIC_FEATURES + cat_names

print("Transformed shape:", X_transformed.shape)
print(f"Total features: {len(feature_names)}")
print("\nAll feature names:", feature_names)

## 3) Model pipelines

We evaluate 6 models: from a simple linear baseline to gradient boosting methods.  
All share the same preprocessing — only the estimator changes.

**Why these models?**
- **LogisticRegression** — linear baseline; surprisingly competitive here because Sex×Pclass is a near-linear signal
- **RandomForest** — bags many trees; lower variance than a single tree
- **GradientBoosting** — sequential error-correction; sklearn's classic implementation
- **HistGB** — histogram-based variant of GB; faster and supports native early stopping
- **XGBoost** — industry standard; excellent regularization options, highly tuneable
- **LightGBM** — leaf-wise growth instead of level-wise; often faster than XGBoost on medium datasets

In [ ]:
def make_pipeline(model):
    return Pipeline([("preprocess", preprocess), ("model", model)])

pipelines = {
    "LogisticRegression": make_pipeline(
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    ),
    "RandomForest": make_pipeline(
        RandomForestClassifier(n_estimators=400, min_samples_leaf=2,
                               n_jobs=-1, random_state=RANDOM_STATE)
    ),
    "GradientBoosting": make_pipeline(
        GradientBoostingClassifier(n_estimators=300, learning_rate=0.05,
                                   max_depth=3, random_state=RANDOM_STATE)
    ),
    "HistGB": make_pipeline(
        HistGradientBoostingClassifier(learning_rate=0.05, max_bins=255,
                                       early_stopping=True, validation_fraction=0.1,
                                       random_state=RANDOM_STATE)
    ),
    "XGBoost": make_pipeline(
        XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=4,
                      subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                      objective="binary:logistic", tree_method="hist",
                      random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)
    ),
    "LightGBM": make_pipeline(
        LGBMClassifier(n_estimators=400, learning_rate=0.05, max_depth=4,
                       subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                       random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
    ),
}

print("Models to evaluate:", list(pipelines.keys()))

## 4) Cross-validation benchmark

We use **Stratified K-Fold** (5 folds) and report three metrics:
- **Accuracy** — the Kaggle leaderboard metric
- **ROC-AUC** — measures discrimination ability regardless of threshold; better for imbalanced classes
- **F1** — harmonic mean of precision and recall; penalizes models that ignore the minority class

Reporting only accuracy on a 62/38 imbalanced dataset can be misleading — a model that always predicts "died" would still get 61.6%.

In [ ]:
skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

cv_results = {}

for name, pipe in pipelines.items():
    acc = cross_val_score(pipe, X, y, cv=skf, scoring="accuracy")
    auc = cross_val_score(pipe, X, y, cv=skf, scoring="roc_auc")
    f1  = cross_val_score(pipe, X, y, cv=skf, scoring="f1")
    cv_results[name] = {"accuracy": acc, "roc_auc": auc, "f1": f1}
    print(f"{name:20s}  Acc={acc.mean():.4f}\u00b1{acc.std():.4f}  "
          f"AUC={auc.mean():.4f}\u00b1{auc.std():.4f}  "
          f"F1={f1.mean():.4f}\u00b1{f1.std():.4f}")

### 4.1) Model comparison table

In [ ]:
comparison = pd.DataFrame([
    {
        "Model":            name,
        "Accuracy (mean)": cv_results[name]["accuracy"].mean().round(4),
        "Accuracy (std)":  cv_results[name]["accuracy"].std().round(4),
        "ROC-AUC (mean)": cv_results[name]["roc_auc"].mean().round(4),
        "F1 (mean)":       cv_results[name]["f1"].mean().round(4),
    }
    for name in cv_results
])

comparison = comparison.sort_values("ROC-AUC (mean)", ascending=False).reset_index(drop=True)
display(comparison)

### 4.2) Holdout set — confusion matrix & ROC curve

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

# Best model by ROC-AUC
best_name = comparison.iloc[0]["Model"]
best_pipe = pipelines[best_name]
best_pipe.fit(X_train, y_train)

val_preds = best_pipe.predict(X_val)
val_proba = best_pipe.predict_proba(X_val)[:, 1]

print(f"Best model: {best_name}")
print(f"Holdout Accuracy : {accuracy_score(y_val, val_preds):.4f}")
print(f"Holdout ROC-AUC  : {roc_auc_score(y_val, val_proba):.4f}")
print(f"Holdout F1       : {f1_score(y_val, val_preds):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(y_val, val_preds, labels=[0, 1])
disp = ConfusionMatrixDisplay(cm, display_labels=['Died', 'Survived'])
disp.plot(ax=axes[0])
axes[0].set_title(f"Confusion Matrix — {best_name}")

RocCurveDisplay.from_predictions(y_val, val_proba, ax=axes[1], name=best_name)
axes[1].set_title("ROC Curve")

plt.tight_layout()
plt.show()

## 5) Hyperparameter tuning (XGBoost)

We tune XGBoost with a random search over 30 configurations, optimizing for **ROC-AUC** instead of accuracy.  
Key parameters to tune:
- `learning_rate` + `n_estimators` — tradeoff between step size and number of steps
- `max_depth` — controls tree complexity; deeper = more expressive but also more prone to overfit
- `subsample` + `colsample_bytree` — introduce randomness to reduce variance (like Random Forest)
- `reg_lambda` — L2 regularization; penalizes large leaf weights

In [ ]:
xgb_param_dist = {
    "model__n_estimators":     randint(200, 800),
    "model__learning_rate":    loguniform(1e-3, 3e-1),
    "model__max_depth":        randint(3, 7),
    "model__subsample":        loguniform(0.5, 1.0),
    "model__colsample_bytree": loguniform(0.5, 1.0),
    "model__reg_lambda":       loguniform(1e-2, 10),
    "model__min_child_weight": randint(1, 10)
}

xgb_search = RandomizedSearchCV(
    estimator=pipelines["XGBoost"],
    param_distributions=xgb_param_dist,
    n_iter=30,
    scoring="roc_auc",
    cv=skf,
    refit=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X, y)
print("Best XGB CV ROC-AUC:", round(xgb_search.best_score_, 4))
print("Best XGB params:", xgb_search.best_params_)
best_xgb_pipeline = xgb_search.best_estimator_

## 6) Model interpretation with SHAP

SHAP (SHapley Additive exPlanations) answers: *why did the model make this prediction?*

Each feature gets a SHAP value per passenger. The SHAP value represents **how much that feature pushed the prediction up or down** relative to the average prediction.

- **Positive SHAP** → pushed toward "survived"
- **Negative SHAP** → pushed toward "died"
- The sum of all SHAP values equals the final model output (minus base rate)

We use `TreeExplainer`, which computes exact SHAP values for tree-based models efficiently.

In [ ]:
# Extract the fitted XGBoost model and preprocessor from the tuned pipeline
xgb_model = best_xgb_pipeline.named_steps['model']
X_proc    = best_xgb_pipeline.named_steps['preprocess'].transform(X)

# Reconstruct feature names after one-hot encoding
ohe_fitted       = best_xgb_pipeline.named_steps['preprocess'].named_transformers_['cat'].named_steps['onehot']
cat_names_fitted = ohe_fitted.get_feature_names_out(CATEGORICAL_FEATURES).tolist()
all_feature_names = NUMERIC_FEATURES + cat_names_fitted

# Build SHAP explainer and compute values
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_proc)

print("SHAP values shape:", shap_values.shape)
print("One row per passenger, one column per feature — same as X_proc.")

In [ ]:
# Beeswarm plot — global feature importance
# Each dot = one passenger. Color = feature value. X-axis = impact on model output.
# Features sorted by mean |SHAP| (most impactful at the top)
shap.summary_plot(shap_values, X_proc, feature_names=all_feature_names, max_display=20)

In [ ]:
# Bar plot — mean absolute SHAP per feature (clean importance ranking)
shap.summary_plot(shap_values, X_proc, feature_names=all_feature_names,
                  plot_type="bar", max_display=20)

In [ ]:
# Waterfall plot — individual passenger explanation
# Let's look at passenger index 0: a 3rd class male who died
passenger_idx = 0
print(f"Passenger {passenger_idx}:")
print(X.iloc[passenger_idx][['Sex', 'Pclass', 'Age', 'Fare', 'Embarked']].to_dict())
print(f"Actual: {'Survived' if y.iloc[passenger_idx] == 1 else 'Died'}")
print()

shap_exp = shap.Explanation(
    values=shap_values[passenger_idx],
    base_values=explainer.expected_value,
    data=X_proc[passenger_idx],
    feature_names=all_feature_names
)
shap.waterfall_plot(shap_exp)

## 7) Train on full data & create submissions

Two submissions:
1. **Best model by CV ROC-AUC** — the most consistent performer across folds
2. **Tuned XGBoost** — best after hyperparameter search

In [ ]:
# Submission 1 — Best CV model (full retrain)
final_name = comparison.iloc[0]["Model"]
final_pipe = pipelines[final_name]
final_pipe.fit(X, y)
pipe_pred  = final_pipe.predict(X_test_raw).astype(int)

sub_pipe = pd.DataFrame({"PassengerId": X_test_raw["PassengerId"], "Survived": pipe_pred})
sub_pipe_path = os.path.join(OUTPUT_DIR, "submission_best_cv.csv")
sub_pipe.to_csv(sub_pipe_path, index=False)
print(f"Saved: {sub_pipe_path}  (model: {final_name})")

# Submission 2 — Tuned XGBoost
xgb_pred = best_xgb_pipeline.predict(X_test_raw).astype(int)
sub_xgb  = pd.DataFrame({"PassengerId": X_test_raw["PassengerId"], "Survived": xgb_pred})
sub_xgb_path = os.path.join(OUTPUT_DIR, "submission_tuned_xgb.csv")
sub_xgb.to_csv(sub_xgb_path, index=False)
print(f"Saved: {sub_xgb_path}  (tuned XGBoost)")

## 8) What I learned

A few things that stood out once I could interpret the model with SHAP:

**`Title_Mr` was the single strongest predictor** — more than `Sex_female` directly.  
This makes sense: "Mr" encodes being male, adult, and without a special honorific all at once. The model can use it as a single strong signal instead of combining three weaker ones.

**`WomanOrChild` shows up clearly** in the SHAP beeswarm.  
The explicit encoding of the rescue protocol was useful — the model didn't need to infer it from a combination of `Sex` and `Age`.

**`FarePerPerson` and `TicketGroupSize` contributed**, even if modestly.  
People traveling in groups beyond their declared family — and who paid more per head — had slightly better odds. Possibly because wealthier group travelers were in better deck locations.

**Boosting vs. simpler models:** Logistic Regression is competitive here because the main signal (Sex × Pclass) is nearly linear. The boosted models extract more from the tail features (Deck, Embarked, Title).  
This is a useful reminder that not every tabular problem needs a complex model.

**What I'd do next:**
- Try a soft-voting ensemble (LR + XGBoost + LightGBM)
- Better Age imputation: grouped median by Title × Pclass × Sex instead of global median
- Explore LightGBM native categorical support (feed `AgeGroup`, `Deck` without one-hot encoding)